In [3]:
from sqlalchemy import (
    Table, MetaData, Column, Integer, String, 
    create_engine, ForeignKey, Date, Time, Enum
)

engine = create_engine('postgresql:///premier_league')

metadata = MetaData()

competition_table = Table(
    'competition', metadata,
    Column('idcompetition', Integer, primary_key=True),
    Column('nomcompetition', String(155))
)

saison_table = Table(
    'saison', metadata,
    Column('id_saison', Integer, primary_key=True),
    Column('annee', Integer)
)

equipe_table = Table(
    'equipe', metadata,
    Column('idequipe', Integer, primary_key=True),
    Column('nomequipe', String(250)),
    Column('idcompetition', Integer, ForeignKey('competition.idcompetition')),
    Column('idsaison', Integer, ForeignKey('saison.id_saison'))
)

joueur_table = Table(
    'joueur', metadata,
    Column('idjoueur', Integer, primary_key=True),
    Column('nomjoueur', String(250)),
    Column('position', String(100)),
    Column('nationalite', String(100)),
    Column('id_equipe', Integer, ForeignKey('equipe.idequipe'))
)

match_table = Table(
    'match', metadata,
    Column('idmatch', Integer, primary_key=True),
    Column('date_match', Date),
    Column('heure', Time),
    Column('round', String(100)),
    Column('venue', String(250)),
    Column('idteamhome', Integer, ForeignKey('equipe.idequipe')),
    Column('idteam_away', Integer, ForeignKey('equipe.idequipe')),
    Column('id_competition', Integer, ForeignKey('competition.idcompetition')),
    Column('id_saison', Integer, ForeignKey('saison.id_saison'))
)

resultatmatch_table = Table(
    'resultatmatch', metadata,
    Column('idresultat', Integer, primary_key=True),
    Column('idmatch', Integer, ForeignKey('match.idmatch')),
    Column('idequipe', Integer, ForeignKey('equipe.idequipe')),
    Column('butsmarques', Integer),
    Column('butsconcedes', Integer),
    Column('resultat', Enum('Victoire', 'Défaite', 'Nul', name='resultat_enum', create_type=True))
)

statistiquejoueur_table = Table(
    'statistiquejoueur', metadata,
    Column('idstats', Integer, primary_key=True),
    Column('idjoueur', Integer, ForeignKey('joueur.idjoueur')),
    Column('buts', Integer, default=0),
    Column('passesdecisives', Integer, default=0),
    Column('nbmatchesplayed', Integer, default=0),
    Column('cartonsjaunes', Integer, default=0),
    Column('cartonsrouges', Integer, default=0)
)

if __name__ == "__main__":
    try:
        metadata.create_all(engine)
        print("success")
    except Exception as e:
        print(f"error: {e}")
        

success


In [8]:
from sqlalchemy import insert, select
from sqlalchemy import create_engine
import pandas as pd


engine = create_engine('postgresql:///premier_league')
df = pd.read_csv('premier_league_matchs.csv')

with engine.begin() as conn:
    competitions = df['comp'].unique()
    comp_dict = {}

    for i, name in enumerate(competitions, start=1):
        conn.execute(insert(competition_table).values(idcompetition=i, nomcompetition=name))
        comp_dict[name] = i

    conn.execute(insert(saison_table).values(id_saison=1, annee=2024))

    all_teams = sorted(set(df['team']).union(df['opponent']))
    premier_league_id = comp_dict.get('Premier League', 1)

    equipes_data = [
        dict(idequipe=i, nomequipe=team, idcompetition=premier_league_id, idsaison=1)
        for i, team in enumerate(all_teams, start=1)
    ]
    conn.execute(insert(equipe_table), equipes_data)

    print(f"inserted: {len(competitions)} competitions, 1 saison, {len(all_teams)} equipes")
    for c in competitions:
        print(f"  - {c}")


inserted: 7 competitions, 1 saison, 82 equipes
  - Premier League
  - EFL Cup
  - Champions Lg
  - FA Cup
  - FA Community Shield
  - Conf Lg
  - Europa Lg


In [5]:
df

,player,nationality,position,age,games,games_starts,minutes,minutes_90s,goals,assists,goals_assists,goals_pens,pens_made,pens_att,cards_yellow,cards_red,team
0,Mohamed Salah,eg EGY,FW,32.0,38,38,"3,371",37.5,29.0,18.0,47.0,20.0,9.0,9.0,1.0,0.0,Liverpool
1,Virgil van Dijk,nl NED,DF,33.0,37,37,"3,330",37.0,3.0,1.0,4.0,3.0,0.0,0.0,5.0,0.0,Liverpool
2,Ryan Gravenberch,nl NED,MF,22.0,37,37,"3,160",35.1,0.0,4.0,4.0,0.0,0.0,0.0,6.0,1.0,Liverpool
3,Alexis Mac Allister,ar ARG,MF,25.0,35,30,"2,599",28.9,5.0,5.0,10.0,5.0,0.0,0.0,6.0,0.0,Liverpool
4,Ibrahima Konaté,fr FRA,DF,25.0,31,30,"2,560",28.4,1.0,2.0,3.0,1.0,0.0,0.0,5.0,0.0,Liverpool
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
697,Joachim Kayi Sanda,fr FRA,DF,17.0,2,0,14,0.2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Southampton
698,Ronnie Edwards,eng ENG,DF,21.0,1,0,12,0.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Southampton
699,Carlos Alcaraz,ar ARG,MF,21.0,1,0,10,0.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Southampton
700,Jayden Moore,eng ENG,DF,18.0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Southampton
